# S5 · Ejecuta el loop y míralo en LangSmith

**Claude para Productividad · Nivel 2 · León Ruiz / Collective Academy**

Al terminar verás en tu propia cuenta una traza nueva con cinco vueltas del ciclo:

> `redactar → medir → evaluar → decidir → repetir`

**No tienes que programar ni cambiar el código.** Solo necesitas una cuenta de LangSmith y tu propia API key.

> Google Colab ejecuta el código. LangSmith no es el lugar donde vive el agente: es el lugar donde observarás lo que hizo.


## Antes de presionar “Ejecutar todo”

1. Abre [LangSmith Settings](https://smith.langchain.com/settings) e inicia sesión.
2. Ve a **API Keys → Create API Key**.
3. Crea una llave personal, cópiala y vuelve a esta pestaña. LangSmith solo la muestra una vez.[1]
4. En Colab, abre **Entorno de ejecución → Ejecutar todas**.
5. Cuando aparezca el campo `Pega tu API key de LangSmith`, pégala y presiona Enter. El texto permanecerá oculto.

La actividad utiliza borradores ficticios ya generados. **No necesita OpenRouter, no consume llamadas nuevas a Claude y no envía correos.**


In [ ]:
# PASO 1 · Preparar el ejercicio automáticamente. No edites esta celda.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nabolom/curso-claude-productividad-s5.git"
REPO_DIR = Path("/content/curso-claude-productividad-s5")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "langgraph==1.2.11",
        "langchain-openai==1.6.2",
        "langsmith==0.12.6",
        "python-dotenv==1.2.3",
    ],
    check=True,
)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("✅ Ejercicio preparado. Continúa con el campo que aparece abajo.")


In [ ]:
# PASO 2 · Pega tu propia llave. No se mostrará en pantalla ni se guardará en el notebook.
import getpass
from langsmith import Client

langsmith_key = getpass.getpass("Pega tu API key de LangSmith y presiona Enter: ").strip()
if not langsmith_key:
    raise ValueError("No se recibió una llave. Vuelve a ejecutar esta celda y pégala.")

os.environ["LANGSMITH_API_KEY"] = langsmith_key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "S5-HillClimbing-Mi-Traza"
del langsmith_key

try:
    next(Client().list_projects(limit=1), None)
except Exception as error:
    os.environ.pop("LANGSMITH_API_KEY", None)
    raise ValueError(
        "LangSmith no aceptó la llave. Crea una Personal Access Token nueva, "
        "vuelve a ejecutar esta celda y pégala sin espacios."
    ) from error

print("✅ LangSmith configurado. La siguiente celda ejecutará el loop.")


## Ahora observa la ejecución

La siguiente celda hará cinco iteraciones. Al terminar imprimirá:

- la secuencia de puntajes;
- el mensaje ganador;
- un enlace directo a **tu propia traza** en LangSmith.

El proceso puede tardar unos segundos mientras LangSmith recibe la traza.


In [ ]:
# PASO 3 · Ejecutar el loop, esperar la traza y mostrar su enlace. No edites esta celda.
import time
from datetime import datetime, timedelta, timezone

from langchain_core.tracers.langchain import wait_for_all_tracers
from langsmith import Client

from demo_hill_climbing import ejecutar_demo, imprimir_resumen

PROJECT_NAME = os.environ["LANGSMITH_PROJECT"]
inicio = datetime.now(timezone.utc) - timedelta(seconds=5)

resultado = ejecutar_demo(modo="replay", max_iteraciones=5)
imprimir_resumen(resultado)
wait_for_all_tracers()

cliente = Client()
try:
    proyecto = None
    ultimo_error = None
    for _ in range(8):
        try:
            proyectos = list(cliente.list_projects(name=PROJECT_NAME, limit=1))
            if proyectos:
                proyecto = proyectos[0]
                break
        except Exception as error:
            ultimo_error = error
        time.sleep(2)

    if proyecto is None:
        raise RuntimeError(
            "La ejecución terminó, pero el proyecto todavía no aparece en LangSmith. "
            "Espera 10 segundos y vuelve a ejecutar PASO 2 y después PASO 3."
        ) from ultimo_error

    corridas = []
    for _ in range(8):
        try:
            pagina = cliente.runs.query(
                project_ids=[str(proyecto.id)],
                is_root=True,
                min_start_time=inicio,
                page_size=10,
                selects=["ID", "NAME", "PROJECT_ID", "TRACE_ID", "START_TIME"],
            )
            corridas = [run async for run in pagina]
        except Exception as error:
            ultimo_error = error
        if corridas:
            break
        time.sleep(2)

    if not corridas:
        raise RuntimeError(
            "La ejecución terminó, pero LangSmith todavía no devolvió la traza. "
            "Espera 10 segundos y vuelve a ejecutar PASO 2 y después PASO 3."
        ) from ultimo_error

    candidatas = [
        run for run in corridas if run.name == "S5 Hill Climbing Reactivacion"
    ] or corridas
    corrida = max(candidatas, key=lambda run: run.start_time)
    enlace = await cliente.runs.get_url(
        str(corrida.id),
        project_id=str(proyecto.id),
        trace_id=str(corrida.trace_id or corrida.id),
        start_time=corrida.start_time.isoformat(),
    )

    print("\n" + "=" * 72)
    print("✅ TU TRAZA ESTÁ LISTA")
    print("Haz clic en este enlace:")
    print(enlace.url)
finally:
    os.environ.pop("LANGSMITH_API_KEY", None)
    del cliente
    print("🔒 La llave se retiró de la memoria del notebook.")


## Qué debes encontrar en LangSmith

Abre el enlace que imprimió la celda anterior y busca el patrón que se repite cinco veces:

1. **`redactar`** toma el siguiente borrador de la corrida de referencia.
2. **`medir`** aplica la misma rúbrica a ese borrador.
3. **`evaluar`** decide si el intento supera el mejor puntaje previo.
4. **`ruta_decision`** elige terminar o volver a `redactar`.

Localiza la progresión **5 → 50 → 62 → 62 → 70**. La cuarta ronda se descarta porque no supera el récord de 62.

### Responde antes de cerrar

- ¿Qué parte del sistema genera una opción?
- ¿Qué parte decide si esa opción es mejor?
- ¿Por qué una ronda sin mejora sigue siendo evidencia útil?
- ¿Qué podría salir mal si la métrica premiara algo equivocado?

> En esta ruta, el grafo y la traza son nuevos; los cinco textos son una reproducción transparente de una corrida previa de Claude. La extensión avanzada del repositorio permite generar textos nuevos con OpenRouter.

## Referencias

[1]: https://docs.langchain.com/langsmith/create-account-api-key "Create an account and API key · LangSmith Docs"
[2]: https://docs.langchain.com/langsmith/observability-quickstart "LangSmith observability quickstart"
